# APGCC — Re-implementation
**Improving Point-based Crowd Counting and Localization Based on Auxiliary Point Guidance** (Chen et al., ECCV 2024, arXiv:2405.10589)
Official repo: https://github.com/AaronCIH/APGCC

This notebook re-implements the full method in one self-contained PyTorch file-equivalent:

| Section | Paper reference |
|---|---|
| 1. Config (all hyper-parameters) | §5.3 Training Details |
| 2. VGG16-BN encoder | §4.3, Fig. 4 |
| 3. ASPP | §4.3 |
| 4. Implicit Feature Interpolation (IFI) | §4.2, Eq. (8), Fig. 3 |
| 5. Reference (anchor) points | §3 Point Proposal Prediction |
| 6. IFI decoder + shared prediction heads | §4.3 |
| 7. **Auxiliary Point Guidance (APG)** forward | §4.1, Fig. 2 |
| 8. Hungarian matcher | §3, Eq. (1) |
| 9. Losses: $\mathcal{L}_{point}$, $\mathcal{L}_{APG}$, $\mathcal{L}_{overall}$ | Eq. (2)–(7), (9) |
| 10. Dataset + augmentation (SHHA format) | §5.1, §5.3 |
| 11. Inference + metrics (MAE/MSE, P/R/F1 @ σ) | §5.2 |
| 12. Instability Rate (IR) | Fig. 1 (right) |
| 13. Training loop (defined, **not run**) | §5.3 |
| 14. Sanity checks (shapes, losses, gradients) | — |

### Where the official repo and the paper differ (and what this notebook does)
1. **APG forward is not in the repo.** `IFI_Decoder_Model.forward` ends with `raise NotImplemented  # still refinement` when `AUX_EN=True`; the loss (`loss_auxiliary`) exists but nothing produces its inputs. Section 7 implements it from §4.1.
2. **IFI weighted sum bug.** The repo's local branch writes `x = ret + conte * w` inside the loop, so only the *last* neighbour survives. Combined with point 3, at interior cells the conv3 (H/8) branch ends up with weight ≈1e-9, so the released model effectively uses only the conv4+ASPP features (verified numerically against the repo code). Eq. (8) is a sum over all 4 neighbours. `ifi_mode='paper'` (default) does the real sum; `ifi_mode='repo'` reproduces the repo **bit-for-bit** (checked: identical outputs to the official `Model_builder` with shared weights) — use it with the official checkpoint.
3. **Neighbour search radius.** Paper: "four nearest latent features". Repo shifts by half a cell of the *target* grid (H/8) for every level, which for the H/16 level does not pick the 4 nearest cells. `'paper'` mode shifts by half a cell of each feature map (LIIF-style); `'repo'` mode copies the repo.
4. **Where the MLP sits.** Eq. (8) writes the area-weighted sum *outside* $f_\theta$; the repo blends the inputs `[φ(δ), Z]` and runs one shared MLP head on the concatenation of both levels. I follow the repo (it is what the released weights use, and it's ~4× cheaper).
5. Eq. (5) is printed without the minus sign on `log ĉ`; it is implemented as a cross-entropy (as in the repo).

## 0. Imports

In [1]:
import math, os, random, glob
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import linear_sum_assignment
from PIL import Image
import torchvision
from torchvision import transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| device', device)

torch 2.15.0+cu132 | torchvision 0.29.2+cu132 | device cpu


## 1. Configuration
Values come from §5.3 and `configs/SHHA_IFI.yml`. Mapping to the paper's symbols:

| Symbol | Meaning | Value |
|---|---|---|
| $s$ | proposal stride | 8 |
| $K$ | reference points per cell (ROW×LINE) | 4 (8 for QNRF) |
| $\gamma$ | offset scale | 100 |
| $\tau$ | distance weight in matching cost | 0.05 |
| $\lambda_1$ | weight of negatives in $\mathcal{L}_{cls}$ | 0.5 |
| $\lambda_2,\lambda_3,\lambda_4$ | loc-loss weights | 2e-4 |
| $\lambda_5$ | APG weight | 0.2 |
| $(k_{pos}, k_{neg})$ | # auxiliary points per GT | (2, 2) |
| $(n_{pos}, n_{neg})$ | randomness ranges (px) | (2, 8) |

In [2]:
@dataclass
class Cfg:
    # model
    pretrained_backbone: bool = True       # ImageNet VGG16-BN
    inner_planes: int = 64                 # channel width after enc3 / ASPP head
    aspp_dilations: tuple = (2, 4, 8)
    pos_dim: int = 32                      # positional-encoding size (ultra_pe); +2 raw coords
    head_layers: tuple = (1024, 512, 256, 256)   # shared prediction head (§5.3)
    stride: int = 8                        # s
    row: int = 2; line: int = 2            # K = row*line
    gamma: float = 100.0                   # γ
    num_classes: int = 2
    ifi_mode: str = 'paper'                # 'paper' or 'repo' (see intro)
    # matcher
    tau: float = 0.05                      # τ  (cost_point)
    cost_class: float = 1.0
    # loss
    lambda1: float = 0.5                   # eos_coef
    lambda2: float = 2e-4                  # point loss
    lambda3: float = 2e-4                  # APG pos loc
    lambda4: float = 2e-4                  # APG neg loc
    lambda5: float = 0.2                   # APG total weight
    aux_enabled: bool = True
    k_pos: int = 2; k_neg: int = 2
    n_pos: float = 2.0; n_neg: float = 8.0
    # optimisation / data
    lr: float = 1e-4; lr_backbone: float = 1e-5; weight_decay: float = 1e-4
    clip_max_norm: float = 0.1
    batch_size: int = 8; crop_size: int = 128; crop_number: int = 4
    scale_range: tuple = (0.7, 1.3)
    epochs: int = 3500
    test_threshold: float = 0.5

    @property
    def K(self): return self.row * self.line

cfg = Cfg()
cfg

Cfg(pretrained_backbone=True, inner_planes=64, aspp_dilations=(2, 4, 8), pos_dim=32, head_layers=(1024, 512, 256, 256), stride=8, row=2, line=2, gamma=100.0, num_classes=2, ifi_mode='paper', tau=0.05, cost_class=1.0, lambda1=0.5, lambda2=0.0002, lambda3=0.0002, lambda4=0.0002, lambda5=0.2, aux_enabled=True, k_pos=2, k_neg=2, n_pos=2.0, n_neg=8.0, lr=0.0001, lr_backbone=1e-05, weight_decay=0.0001, clip_max_norm=0.1, batch_size=8, crop_size=128, crop_number=4, scale_range=(0.7, 1.3), epochs=3500, test_threshold=0.5)

## 2. Encoder — VGG16-BN
Split into 4 stages with strides 2, 4, 8, 16 (no final pooling). Only stages 3 (`conv3`, H/8, 512ch) and 4 (`conv4`, H/16, 512ch) feed the decoder (Fig. 4). Slice indices are those of `Base_VGG` in the repo.

In [3]:
class Base_VGG(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.VGG16_BN_Weights.IMAGENET1K_V1 if pretrained else None
        feats = list(torchvision.models.vgg16_bn(weights=weights).features.children())
        self.body1 = nn.Sequential(*feats[:13])    # /2  , 128ch
        self.body2 = nn.Sequential(*feats[13:23])  # /4  , 256ch
        self.body3 = nn.Sequential(*feats[23:33])  # /8  , 512ch
        self.body4 = nn.Sequential(*feats[33:43])  # /16 , 512ch  (last max-pool dropped)
        self.out_planes = [128, 256, 512, 512]

    def forward(self, x):
        outs = []
        for body in (self.body1, self.body2, self.body3, self.body4):
            x = body(x); outs.append(x)
        return outs

## 3. ASPP
Atrous Spatial Pyramid Pooling on the deepest feature for scale diversity (§4.3). Same configuration as the repo: no BN inside, dilations (2, 4, 8), 5 branches → `5 × inner_planes` channels.

In [4]:
class ASPP(nn.Module):
    def __init__(self, in_planes, inner_planes=256, dilations=(2, 4, 8)):
        super().__init__()
        def br(k, d):
            return nn.Sequential(nn.Conv2d(in_planes, inner_planes, k, padding=0 if k == 1 else d,
                                           dilation=d, bias=False), nn.ReLU(inplace=True))
        self.conv1 = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                   nn.Conv2d(in_planes, inner_planes, 1, bias=False), nn.ReLU(inplace=True))
        self.conv2 = br(1, 1)
        self.conv3 = br(3, dilations[0])
        self.conv4 = br(3, dilations[1])
        self.conv5 = br(3, dilations[2])
        self.out_planes = (len(dilations) + 2) * inner_planes

    def get_outplanes(self): return self.out_planes

    def forward(self, x):
        h, w = x.shape[-2:]
        f1 = F.interpolate(self.conv1(x), size=(h, w), mode='bilinear', align_corners=True)
        return torch.cat([f1, self.conv2(x), self.conv3(x), self.conv4(x), self.conv5(x)], 1)

## 4. Implicit Feature Interpolation (IFI)
For a query at continuous position $(x,y)$ (Fig. 3, Eq. 8):
1. find the 4 nearest latent vectors $Z^*_i$ (nearest-neighbour `grid_sample` after shifting the query by ±½ cell in each axis);
2. compute relative offsets $\delta^*_i$ (in feature-cell units) and encode them, $\phi(\delta^*_i)=[\delta, \sin(\delta M), \cos(\delta M)]$ with a learnable frequency matrix $M$ (`SpatialEncoding`, `ultra_pe=True` in the repo);
3. blend with bilinear-style weights $S_i/S$, where $S_i$ is the area of the rectangle between the query and the **diagonally opposite** neighbour.

Coordinates are normalised to $[-1,1]$ in **(y, x)** order with the `align_corners=False` convention, as in the repo. `implicit_sample` works for *any* query coordinates — that is what makes APG possible, because auxiliary points lie off the grid.

In [5]:
class SpatialEncoding(nn.Module):
    # φ(δ): δ (2-d) -> [δ, sin(δM^T), cos(δM^T)], out = 2 + out_dim. Frequencies 2^linspace(0, σ).
    def __init__(self, in_dim=2, out_dim=32, sigma=6, learnable=True):
        super().__init__()
        assert out_dim % (2 * in_dim) == 0
        n = out_dim // 2 // in_dim
        m = 2 ** np.linspace(0, sigma, n)
        m = np.stack([m] + [np.zeros_like(m)] * (in_dim - 1), axis=-1)
        m = np.concatenate([np.roll(m, i, axis=-1) for i in range(in_dim)], axis=0)  # (out_dim/2, in_dim)
        emb = torch.tensor(m, dtype=torch.float32)
        if learnable: self.emb = nn.Parameter(emb)
        else:         self.register_buffer('emb', emb)
        self.out_dim = out_dim + in_dim

    def forward(self, x):
        y = x @ self.emb.T
        return torch.cat([x, torch.sin(y), torch.cos(y)], dim=-1)


def grid_query_coords(h, w, device, roundtrip=False):
    # (h*w, 2) normalised (y, x) centres of an h×w grid. Arithmetic copied from the repo's make_coord
    # (and its [0,1] round-trip for queries) so float rounding matches bit-for-bit — this matters at the
    # image border, where IFI area weights are ratios of near-zero numbers.
    seqs = []
    for n in (h, w):
        r = 2 / (2 * n)
        seqs.append(-1 + r + (2 * r) * torch.arange(n, device=device).float())
    yy, xx = torch.meshgrid(*seqs, indexing='ij')
    c = torch.stack([yy, xx], -1).view(-1, 2)
    if roundtrip:
        c = (((c.flip(-1) + 1) / 2) * 2 - 1).flip(-1)
    return c


def implicit_sample(feat, coords, pe, mode='paper', target_hw=None, eps=1e-6):
    # feat:   (B, C, hh, ww) latent map
    # coords: (B, N, 2) normalised (y, x) query positions in [-1, 1]
    # pe:     positional encoder φ
    # returns (B, N, C + pe.out_dim):  Σ_i (S_i/S) · [φ(δ_i), Z_i]
    B, C, hh, ww = feat.shape
    centres = grid_query_coords(hh, ww, feat.device).view(hh, ww, 2).permute(2, 0, 1)
    centres = centres.unsqueeze(0).expand(B, -1, -1, -1)                   # (B, 2, hh, ww)
    if mode == 'paper':
        ry, rx = 1.0 / hh, 1.0 / ww                    # ½ feature cell  → true 4 nearest neighbours
    else:
        th, tw = target_hw
        ry, rx = 1.0 / th, 1.0 / tw                    # repo: ½ cell of the target (H/8) grid

    rels, qfeats, areas = [], [], []
    for vy in (-1, 1):
        for vx in (-1, 1):
            c = coords.clone()
            c[..., 0] += vy * ry + eps
            c[..., 1] += vx * rx + eps
            c.clamp_(-1 + 1e-6, 1 - 1e-6)
            grid = c.flip(-1).unsqueeze(1)             # grid_sample wants (x, y)
            q_feat = F.grid_sample(feat, grid, mode='nearest', align_corners=False)[:, :, 0].permute(0, 2, 1)
            q_coord = F.grid_sample(centres, grid, mode='nearest', align_corners=False)[:, :, 0].permute(0, 2, 1)
            rel = coords - q_coord                       # δ, then scale to cell units
            rel = torch.stack([rel[..., 0] * hh, rel[..., 1] * ww], -1)
            rels.append(rel); qfeats.append(q_feat)
            areas.append((rel[..., 0] * rel[..., 1]).abs() + 1e-9)

    total = torch.stack(areas).sum(0)
    areas = areas[::-1]                                  # diagonal area swap (0↔3, 1↔2)
    ctx = [torch.cat([pe(r), q], -1) for r, q in zip(rels, qfeats)]
    if mode == 'paper':
        return sum(x * (a / total).unsqueeze(-1) for x, a in zip(ctx, areas))
    # repo-faithful: loop overwrites, only the last neighbour survives
    return ctx[-1] * (areas[-1] / total).unsqueeze(-1)

## 5. Reference points
Each cell of the H/8 map owns $K$ reference points $R_k$ laid out on a ROW×LINE sub-grid. A proposal is $\hat p_j = R_k + \gamma\Delta_j$ (§3). Ordering is pixel-major then $k$, matching the head's output layout.

In [6]:
def anchor_shifts(stride, row, line, device):
    # (K, 2) (x, y) offsets of the K reference points from the cell centre; k = r*line + c
    sx = (torch.arange(1, line + 1, device=device) - 0.5) * (stride / line) - stride / 2
    sy = (torch.arange(1, row + 1, device=device) - 0.5) * (stride / row) - stride / 2
    yy, xx = torch.meshgrid(sy, sx, indexing='ij')
    return torch.stack([xx.reshape(-1), yy.reshape(-1)], -1).float()

def reference_points(h, w, stride, row, line, device):
    # (h*w*K, 2) image-pixel (x, y) reference points for an h×w feature grid
    cx = (torch.arange(w, device=device) + 0.5) * stride
    cy = (torch.arange(h, device=device) + 0.5) * stride
    yy, xx = torch.meshgrid(cy, cx, indexing='ij')
    centres = torch.stack([xx, yy], -1).view(-1, 1, 2)               # (hw, 1, 2)
    return (centres + anchor_shifts(stride, row, line, device)[None]).view(-1, 2)

print(anchor_shifts(8, 2, 2, 'cpu'))

tensor([[-2., -2.],
        [ 2., -2.],
        [-2.,  2.],
        [ 2.,  2.]])


## 6. IFI decoder + shared prediction heads
* `enc3`: 1×1 conv on conv3 (H/8) → 64ch; `aspp`+`head`: ASPP on conv4 (H/16) → 64ch.
* For every cell of the H/8 grid, IFI pulls a latent vector from **both** levels (each with its own φ), concatenates them → `2 × (64 + 34) = 196` dims, and feeds the shared heads.
* Heads are point-wise MLPs (Conv1d k=1 + BN + ReLU) with hidden sizes [1024, 512, 256, 256] (§5.3), outputting $K\times2$ offsets and $K\times2$ logits.

Module names (`enc3`, `aspp`, `head`, `ifi.pos1`, `ifi.confidence_head`, …) match the repo so `SHHA_best.pth` loads with `strict=True`.

In [7]:
class IFIHeads(nn.Module):
    # Holds φ per level + shared confidence/offset MLPs (repo name: ifi_simfpn).
    def __init__(self, feat_num, feat_dim, pos_dim, K, num_classes, head_layers):
        super().__init__()
        self.K, self.num_classes = K, num_classes
        for lvl in range(feat_num):
            setattr(self, f'pos{lvl+1}', SpatialEncoding(2, pos_dim, learnable=True))
        self.pe_dim = pos_dim + 2
        self.in_dim = feat_num * (feat_dim + self.pe_dim)

        def mlp(out_dim, last_relu):
            layers, src = [], self.in_dim
            for h in head_layers:
                layers += [nn.Conv1d(src, h, 1), nn.BatchNorm1d(h), nn.ReLU()]; src = h
            layers.append(nn.Conv1d(src, out_dim, 1))
            if last_relu: layers.append(nn.ReLU())   # repo applies ReLU to the logits; kept for weight compatibility
            return nn.Sequential(*layers)
        self.confidence_head = mlp(K * num_classes, last_relu=True)
        self.offset_head = mlp(K * 2, last_relu=False)

    def predict(self, ctx):
        # ctx (B, D, N) -> offset (B, N, K, 2), logits (B, N, K, C)
        B, _, N = ctx.shape
        off = self.offset_head(ctx).permute(0, 2, 1).reshape(B, N, self.K, 2)
        cls = self.confidence_head(ctx).permute(0, 2, 1).reshape(B, N, self.K, self.num_classes)
        return off, cls


class IFI_Decoder(nn.Module):
    def __init__(self, cfg: Cfg, in_planes):
        super().__init__()
        self.cfg = cfg
        p = cfg.inner_planes
        self.enc3 = nn.Sequential(nn.Conv2d(in_planes[2], p, 1), nn.BatchNorm2d(p), nn.ReLU(inplace=True))
        self.aspp = ASPP(in_planes[3], inner_planes=p, dilations=cfg.aspp_dilations)
        self.head = nn.Sequential(nn.Conv2d(self.aspp.get_outplanes(), p, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(p), nn.ReLU(inplace=True), nn.Dropout2d(0.1))
        self.ifi = IFIHeads(feat_num=2, feat_dim=p, pos_dim=cfg.pos_dim, K=cfg.K,
                            num_classes=cfg.num_classes, head_layers=cfg.head_layers)

    def latent_maps(self, feats):
        return [self.enc3(feats[2]), self.head(self.aspp(feats[3]))]

    def context(self, maps, coords, target_hw):
        # IFI on every level at `coords` (B,N,2) and concat → (B, D, N)
        ctx = [implicit_sample(m, coords, getattr(self.ifi, f'pos{i+1}'), self.cfg.ifi_mode, target_hw)
               for i, m in enumerate(maps)]
        return torch.cat(ctx, -1).permute(0, 2, 1)

    def forward(self, feats):
        c = self.cfg
        maps = self.latent_maps(feats)
        ht, wt = feats[2].shape[-2:]                                  # H/8 grid = proposal grid
        B = maps[0].shape[0]
        coords = grid_query_coords(ht, wt, maps[0].device, roundtrip=True)[None].expand(B, -1, -1)
        off, cls = self.ifi.predict(self.context(maps, coords, (ht, wt)))
        offset = off.reshape(B, -1, 2) * c.gamma                      # γΔ
        refs = reference_points(ht, wt, c.stride, c.row, c.line, offset.device)
        out = {'pred_logits': cls.reshape(B, -1, c.num_classes),
               'pred_points': refs[None] + offset,                     # p̂ = R + γΔ
               'offset': offset}
        return out, maps, (ht, wt)

## 7. Auxiliary Point Guidance (APG) — forward pass
Implemented from §4.1 / Fig. 2 (absent from the official repo).

For each GT point $p=(x,y)$:
* $k_{pos}$ positives $A_{pos}=p+R_{pos}$, $R_{pos}\sim U(-n_{pos},n_{pos})^2$
* $k_{neg}$ negatives $A_{neg}=p+R_{neg}$, each coordinate $\sim U([-n_{neg},-n_{pos}]\cup[n_{pos},n_{neg}])$

**Treating an auxiliary point as a proposal.** The heads emit $K$ proposals per query, one per reference slot. For an auxiliary point $a$ I pick the slot $k$ whose sub-cell contains $a$ and query IFI at $a - \text{shift}_k$, so that slot $k$'s reference point is **exactly** $a$. Its prediction is then $\hat p^\star = a + \gamma\Delta^\star_k$ — precisely the quantity Eq. (5) pulls towards $p$ (offset ≈ −randomness) and Eq. (6) pushes to $\Delta^\star\to 0$. This reuses the same shared head and IFI as the real proposals, as Fig. 4 shows.

In [8]:
def sample_aux_offsets(n, k, lo, hi, negative, device):
    # (k, n, 2) random offsets. positive: U(-lo, lo); negative: |r|∈U(lo, hi) with random sign.
    if not negative:
        return (torch.rand(k, n, 2, device=device) * 2 - 1) * lo
    mag = lo + torch.rand(k, n, 2, device=device) * (hi - lo)
    sign = torch.randint(0, 2, (k, n, 2), device=device).float() * 2 - 1
    return mag * sign


def apg_forward(decoder: IFI_Decoder, maps, targets, grid_hw):
    # Returns {'pos': [dict]*k_pos, 'neg': [dict]*k_neg}; every dict has
    #   pred_logits (M, C), pred_points (M, 2), offset (M, 2), gt (M, 2) with M = Σ_b N_b.
    c = decoder.cfg
    ht, wt = grid_hw
    Hs, Ws = ht * c.stride, wt * c.stride
    dev = maps[0].device
    shifts = anchor_shifts(c.stride, c.row, c.line, dev)               # (K, 2)

    kinds = [('pos', i) for i in range(c.k_pos)] + [('neg', j) for j in range(c.k_neg)]
    per_img_ctx, meta = [], []                                          # meta: (kind, idx, a, k, gt)
    for b, t in enumerate(targets):
        gt = t['point'].to(dev).float()
        if gt.numel() == 0: continue
        n = gt.shape[0]
        pos_r = sample_aux_offsets(n, c.k_pos, c.n_pos, c.n_neg, False, dev)
        neg_r = sample_aux_offsets(n, c.k_neg, c.n_pos, c.n_neg, True, dev)
        R = torch.cat([pos_r, neg_r], 0)                                # (k_pos+k_neg, n, 2)
        a = (gt[None] + R).reshape(-1, 2)
        a = torch.stack([a[:, 0].clamp(0, Ws - 1e-3), a[:, 1].clamp(0, Hs - 1e-3)], -1)
        # reference slot containing a
        col = ((a[:, 0] % c.stride) / (c.stride / c.line)).floor().long().clamp(0, c.line - 1)
        row = ((a[:, 1] % c.stride) / (c.stride / c.row)).floor().long().clamp(0, c.row - 1)
        k = row * c.line + col
        q = a - shifts[k]                                               # query so that ref_k == a
        q_norm = torch.stack([2 * q[:, 1] / Hs - 1, 2 * q[:, 0] / Ws - 1], -1)   # (y, x) in [-1,1]
        ctx = decoder.context([m[b:b+1] for m in maps], q_norm[None], grid_hw)   # (1, D, M_b)
        per_img_ctx.append(ctx)
        meta.append((a, k, gt.repeat(len(kinds), 1), n))

    out = {'pos': [], 'neg': []}
    if not per_img_ctx:
        return out
    off, cls = decoder.ifi.predict(torch.cat(per_img_ctx, -1))         # one pass through the shared head
    off, cls = off[0], cls[0]                                           # (M, K, 2), (M, K, C)

    # split back by kind
    start = 0
    buckets = {kd: {'pred_logits': [], 'pred_points': [], 'offset': [], 'gt': []} for kd in kinds}
    for a, k, gt_rep, n in meta:
        M = a.shape[0]
        idx = torch.arange(M, device=dev)
        o = off[start:start + M][idx, k] * c.gamma
        l = cls[start:start + M][idx, k]
        for j, kd in enumerate(kinds):
            s = slice(j * n, (j + 1) * n)
            buckets[kd]['pred_logits'].append(l[s]); buckets[kd]['offset'].append(o[s])
            buckets[kd]['pred_points'].append(a[s] + o[s]); buckets[kd]['gt'].append(gt_rep[s])
        start += M
    for (kind, i) in kinds:
        out[kind].append({key: torch.cat(v, 0) for key, v in buckets[(kind, i)].items()})
    return out

### Full model

In [9]:
class APGCC(nn.Module):
    def __init__(self, cfg: Cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = Base_VGG(pretrained=cfg.pretrained_backbone)
        self.decoder = IFI_Decoder(cfg, self.encoder.out_planes)

    def forward(self, images, targets=None):
        feats = self.encoder(images)
        out, maps, grid_hw = self.decoder(feats)
        if self.training and self.cfg.aux_enabled and targets is not None:
            out['aux'] = apg_forward(self.decoder, maps, targets, grid_hw)   # APG is training-only
        return out

## 8. Hungarian matcher — Eq. (1)
$\mathcal{D}(\mathcal P,\hat{\mathcal P}) = \tau\|p_i-\hat p_j\|_2 - \hat c_j$, solved per image with `linear_sum_assignment`.

In [10]:
class HungarianMatcher(nn.Module):
    def __init__(self, cost_class=1.0, cost_point=0.05):
        super().__init__()
        self.cost_class, self.cost_point = cost_class, cost_point

    @torch.no_grad()
    def forward(self, outputs, targets):
        bs, nq = outputs['pred_logits'].shape[:2]
        prob = outputs['pred_logits'].flatten(0, 1).softmax(-1)[:, 1:2]    # ĉ (person)
        pts = outputs['pred_points'].flatten(0, 1)
        tgt = torch.cat([t['point'].to(pts.device).float() for t in targets])
        if tgt.numel() == 0:
            e = torch.empty(0, dtype=torch.int64)
            return [(e, e) for _ in targets]
        C = self.cost_point * torch.cdist(pts, tgt) - self.cost_class * prob
        C = C.view(bs, nq, -1).cpu()
        sizes = [len(t['point']) for t in targets]
        idx = [linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))]
        return [(torch.as_tensor(i, dtype=torch.int64), torch.as_tensor(j, dtype=torch.int64)) for i, j in idx]

## 9. Losses
* $\mathcal L_{cls}$ (Eq. 2): cross-entropy over all proposals; unmatched ones are background with weight $\lambda_1$.
* $\mathcal L_{loc}$ (Eq. 3): squared L2 between matched proposals and GT, normalised by $N$.
* $\mathcal L_{point}=\mathcal L_{cls}+\lambda_2\mathcal L_{loc}$ (Eq. 4)
* $\mathcal L^{pos}_{APG}$ (Eq. 5): CE→1 + $\lambda_3\|p-\hat p^\star_{pos}\|^2$; $\mathcal L^{neg}_{APG}$ (Eq. 6): CE→0 + $\lambda_4\|\Delta^\star_{neg}\|^2$; averaged over $k_{pos}$ / $k_{neg}$.
* $\mathcal L_{overall}=\mathcal L_{point}+\lambda_5\mathcal L_{APG}$ (Eq. 9)

In [11]:
class APGCCCriterion(nn.Module):
    def __init__(self, cfg: Cfg):
        super().__init__()
        self.cfg = cfg
        self.matcher = HungarianMatcher(cfg.cost_class, cfg.tau)
        self.register_buffer('cls_weight', torch.tensor([cfg.lambda1, 1.0]))   # [background, person]
        self.last_indices = None

    def point_losses(self, out, targets, indices):
        logits = out['pred_logits']
        B, M, _ = logits.shape
        bidx = torch.cat([torch.full_like(s, i) for i, (s, _) in enumerate(indices)]).to(logits.device)
        sidx = torch.cat([s for s, _ in indices]).to(logits.device)
        tgt_cls = torch.zeros(B, M, dtype=torch.long, device=logits.device)
        tgt_cls[bidx, sidx] = 1
        l_cls = F.cross_entropy(logits.transpose(1, 2), tgt_cls, self.cls_weight)          # Eq. 2
        n = max(sum(len(t['point']) for t in targets), 1)
        src = out['pred_points'][bidx, sidx]
        tgt = torch.cat([t['point'].to(logits.device).float()[j] for t, (_, j) in zip(targets, indices)])
        l_loc = F.mse_loss(src, tgt, reduction='sum') / n                                   # Eq. 3
        return l_cls, l_loc

    def apg_loss(self, aux):
        c, dev = self.cfg, self.cls_weight.device
        zero = torch.zeros((), device=dev)
        pos = zero.clone()
        for d in aux['pos']:                                                                # Eq. 5
            n = max(d['pred_points'].shape[0], 1)
            ce = F.cross_entropy(d['pred_logits'], torch.ones(len(d['pred_logits']), dtype=torch.long, device=dev))
            pos = pos + ce + c.lambda3 * F.mse_loss(d['pred_points'], d['gt'], reduction='sum') / n
        neg = zero.clone()
        for d in aux['neg']:                                                                # Eq. 6
            n = max(d['offset'].shape[0], 1)
            ce = F.cross_entropy(d['pred_logits'], torch.zeros(len(d['pred_logits']), dtype=torch.long, device=dev))
            neg = neg + ce + c.lambda4 * d['offset'].pow(2).sum() / n
        pos = pos / max(len(aux['pos']), 1)
        neg = neg / max(len(aux['neg']), 1)
        return pos, neg

    def forward(self, out, targets):
        c = self.cfg
        indices = self.matcher(out, targets)
        self.last_indices = indices
        l_cls, l_loc = self.point_losses(out, targets, indices)
        losses = {'loss_cls': l_cls, 'loss_loc': l_loc}
        total = l_cls + c.lambda2 * l_loc                                                   # Eq. 4
        if 'aux' in out:
            pos, neg = self.apg_loss(out['aux'])
            losses.update(loss_apg_pos=pos, loss_apg_neg=neg)
            total = total + c.lambda5 * (pos + neg)                                         # Eq. 7 + 9
        losses['loss'] = total
        return losses

## 10. Dataset (ShanghaiTech format, like the repo)
Expected layout (produced by the repo's `datasets/prepare_label.py`, or `convert_shanghaitech` below):
```
DATA_ROOT/
  train.list   # each line: <image path> <txt path>   (relative to DATA_ROOT)
  test.list
  train_data/images/IMG_1.jpg  train_data/images/IMG_1.txt   # txt: "x y" per head
```
Training augmentation (§5.3): random scale in [0.7, 1.3] (only if the short side stays ≥128), 4 random 128×128 crops per image, horizontal flip with p=0.5. The repo's loader applies flip to all crops of an image together; that's kept.

In [12]:
def convert_shanghaitech(root, part='A'):
    # Write <img>.txt + train.list/test.list from the original .mat files.
    from scipy.io import loadmat
    base = os.path.join(root, f'part_{part}')
    for split, sub in [('train', 'train_data'), ('test', 'test_data')]:
        lines = []
        for img in sorted(glob.glob(os.path.join(base, sub, 'images', '*.jpg'))):
            mat = img.replace('images', 'ground-truth').replace('IMG_', 'GT_IMG_').replace('.jpg', '.mat')
            pts = loadmat(mat)['image_info'][0][0][0][0][0]
            txt = img.replace('.jpg', '.txt')
            np.savetxt(txt, pts, fmt='%.2f')
            lines.append(f'{os.path.relpath(img, base)} {os.path.relpath(txt, base)}')
        with open(os.path.join(base, f'{split}.list'), 'w') as f:
            f.write('\n'.join(lines))


class CrowdDataset(Dataset):
    MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    def __init__(self, root, train=True, cfg: Cfg = cfg, max_size=None):
        self.root, self.train, self.cfg, self.max_size = root, train, cfg, max_size
        with open(os.path.join(root, 'train.list' if train else 'test.list')) as f:
            pairs = [l.split() for l in f if len(l.strip()) > 1]
        self.items = sorted((os.path.join(root, a), os.path.join(root, b)) for a, b in pairs)
        self.tf = T.Compose([T.ToTensor(), T.Normalize(self.MEAN, self.STD)])

    def __len__(self): return len(self.items)

    @staticmethod
    def _load_points(path):
        pts = np.loadtxt(path, ndmin=2) if os.path.getsize(path) > 0 else np.zeros((0, 2))
        return pts[:, :2].astype(np.float32)

    def __getitem__(self, i):
        img_path, gt_path = self.items[i]
        img = self.tf(Image.open(img_path).convert('RGB'))
        pts = torch.from_numpy(self._load_points(gt_path))
        c = self.cfg
        if self.train:
            s = random.uniform(*c.scale_range)
            if s * min(img.shape[1:]) > c.crop_size:
                img = F.interpolate(img[None], scale_factor=s, mode='bilinear', align_corners=False)[0]
                pts = pts * s
            imgs, targets = self._random_crops(img, pts)
            if random.random() > 0.5:
                imgs = imgs.flip(-1)
                for t in targets: t['point'][:, 0] = c.crop_size - t['point'][:, 0]
            return imgs, targets
        # test: optionally cap the long side
        if self.max_size and max(img.shape[1:]) > self.max_size:
            s = self.max_size / max(img.shape[1:])
            img = F.interpolate(img[None], scale_factor=s, mode='bilinear', align_corners=False)[0]
            pts = pts * s
        return img[None], [{'point': pts, 'labels': torch.ones(len(pts), dtype=torch.long)}]

    def _random_crops(self, img, pts):
        c, cs = self.cfg, self.cfg.crop_size
        _, H, W = img.shape
        imgs, targets = [], []
        for _ in range(c.crop_number):
            y0, x0 = random.randint(0, max(H - cs, 0)), random.randint(0, max(W - cs, 0))
            crop = img[:, y0:y0 + cs, x0:x0 + cs]
            crop = F.pad(crop, (0, cs - crop.shape[2], 0, cs - crop.shape[1]))
            m = (pts[:, 0] >= x0) & (pts[:, 0] < x0 + cs) & (pts[:, 1] >= y0) & (pts[:, 1] < y0 + cs)
            p = pts[m] - torch.tensor([x0, y0], dtype=torch.float32)
            imgs.append(crop); targets.append({'point': p.clone(), 'labels': torch.ones(len(p), dtype=torch.long)})
        return torch.stack(imgs), targets


def collate_crops(batch):
    # Each item is (n_crops, 3, h, w) + list of targets; flatten crops into the batch dim.
    imgs = torch.cat([b[0] for b in batch], 0)
    targets = [t for b in batch for t in b[1]]
    return imgs, targets

## 11. Inference and evaluation (§5.2)
* Count = #proposals with softmax person score > 0.5.
* **MAE / MSE** (MSE is the root-mean-square, as in the field).
* **Localization P/R/F1**: one-to-one Hungarian matching between predictions and GT; a pair is a TP if its distance ≤ σ (σ = 4 and 8 for SHHA).

In [13]:
def pad_to_multiple(img, m=16):
    H, W = img.shape[-2:]
    ph, pw = (m - H % m) % m, (m - W % m) % m
    return F.pad(img, (0, pw, 0, ph)), (H, W)

@torch.no_grad()
def predict(model, img, threshold=0.5):
    # img: (1, 3, H, W) normalised. Returns (N, 2) pixel (x, y) points and their scores.
    model.eval()
    x, (H, W) = pad_to_multiple(img.to(next(model.parameters()).device))
    out = model(x)
    scores = out['pred_logits'].softmax(-1)[0, :, 1]
    pts = out['pred_points'][0]
    keep = scores > threshold
    pts, scores = pts[keep], scores[keep]
    inside = (pts[:, 0] < W) & (pts[:, 1] < H)
    return pts[inside].cpu(), scores[inside].cpu()


def localization_counts(pred, gt, sigma):
    if len(pred) == 0 or len(gt) == 0:
        return 0, len(pred), len(gt)
    D = torch.cdist(pred.float(), gt.float()).numpy()
    r, c = linear_sum_assignment(D)
    tp = int((D[r, c] <= sigma).sum())
    return tp, len(pred) - tp, len(gt) - tp     # tp, fp, fn


@torch.no_grad()
def evaluate(model, dataset, threshold=0.5, sigmas=(4, 8)):
    abs_err, sq_err = [], []
    tot = {s: [0, 0, 0] for s in sigmas}
    for i in range(len(dataset)):
        img, tgt = dataset[i]
        gt = tgt[0]['point']
        pts, _ = predict(model, img, threshold)
        abs_err.append(abs(len(pts) - len(gt))); sq_err.append((len(pts) - len(gt)) ** 2)
        for s in sigmas:
            for k, v in enumerate(localization_counts(pts, gt, s)): tot[s][k] += v
    res = {'MAE': float(np.mean(abs_err)), 'MSE': float(np.sqrt(np.mean(sq_err)))}
    for s, (tp, fp, fn) in tot.items():
        P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
        res[f'P@{s}'], res[f'R@{s}'], res[f'F1@{s}'] = P, R, 2 * P * R / max(P + R, 1e-9)
    return res

## 12. Instability Rate (IR) — Fig. 1 (right)
IR per epoch = fraction of GT points whose matched proposal index changed versus the previous epoch. Track it by feeding `criterion.last_indices` after each step with a stable key per sample (e.g. image name + crop id). Note: with random crops the same GT is only comparable if you evaluate IR on a **fixed** set of crops; the helper takes whatever key you give it.

In [14]:
class InstabilityTracker:
    def __init__(self): self.prev, self.curr = {}, {}

    def update(self, keys, indices):
        # keys: one hashable id per target; indices: matcher output for the same batch
        for key, (src, tgt) in zip(keys, indices):
            for s, t in zip(src.tolist(), tgt.tolist()):
                self.curr[(key, t)] = s

    def end_epoch(self):
        common = set(self.prev) & set(self.curr)
        ir = float(np.mean([self.prev[k] != self.curr[k] for k in common])) if common else float('nan')
        self.prev, self.curr = self.curr, {}
        return ir

## 13. Training loop (defined only — not executed here)
Adam, lr 1e-4 (1e-5 for the VGG backbone), weight decay 1e-4, grad-clip 0.1, batch 8 images × 4 crops. Uncomment the last cell to train once the dataset is in place.

In [15]:
def build_optimizer(model, cfg):
    bb = [p for n, p in model.named_parameters() if n.startswith('encoder') and p.requires_grad]
    rest = [p for n, p in model.named_parameters() if not n.startswith('encoder') and p.requires_grad]
    return torch.optim.Adam([{'params': rest, 'lr': cfg.lr},
                             {'params': bb, 'lr': cfg.lr_backbone}], weight_decay=cfg.weight_decay)


def train_one_epoch(model, criterion, loader, optimizer, cfg, tracker=None):
    model.train(); criterion.train()
    logs = {}
    for step, (imgs, targets) in enumerate(loader):
        imgs = imgs.to(device)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        out = model(imgs, targets)
        losses = criterion(out, targets)
        optimizer.zero_grad()
        losses['loss'].backward()
        if cfg.clip_max_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_max_norm)
        optimizer.step()
        for k, v in losses.items(): logs[k] = logs.get(k, 0.0) + v.item()
    return {k: v / max(step + 1, 1) for k, v in logs.items()}


def fit(cfg, data_root, out_dir='./output', eval_every=5):
    os.makedirs(out_dir, exist_ok=True)
    model = APGCC(cfg).to(device)
    criterion = APGCCCriterion(cfg).to(device)
    opt = build_optimizer(model, cfg)
    train_ds, test_ds = CrowdDataset(data_root, True, cfg), CrowdDataset(data_root, False, cfg)
    loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_crops,
                        num_workers=4, drop_last=True)
    best = float('inf')
    for ep in range(cfg.epochs):
        log = train_one_epoch(model, criterion, loader, opt, cfg)
        if (ep + 1) % eval_every == 0:
            res = evaluate(model, test_ds, cfg.test_threshold)
            print(ep, {k: round(v, 4) for k, v in log.items()}, res)
            if res['MAE'] < best:
                best = res['MAE']; torch.save(model.state_dict(), os.path.join(out_dir, 'best.pth'))
    return model

# model = fit(cfg, data_root='/path/to/ShanghaiTech/part_A')

### Loading the official SHHA checkpoint (optional)
Download `SHHA_best.pth` from the repo README. It was trained with the repo's IFI, so switch to `ifi_mode='repo'` to reproduce its numbers (SHHA MAE 48.8 / MSE 76.7 in Table 1).

In [16]:
def load_official(path, ifi_mode='repo'):
    c = Cfg(pretrained_backbone=False, ifi_mode=ifi_mode)
    m = APGCC(c)
    sd = torch.load(path, map_location='cpu')
    sd = sd.get('model', sd)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print('missing:', missing, '\nunexpected:', unexpected)
    return m.to(device).eval()

# model = load_official('./SHHA_best.pth')
# print(evaluate(model, CrowdDataset('/path/to/part_A', train=False)))

## 14. Sanity checks (no training)
Random weights (`pretrained_backbone=False` so nothing is downloaded), synthetic 128×128 crops with random heads. Checks shapes, that APG reference points land exactly on the auxiliary points, that every loss term is finite, and that gradients reach the backbone.

In [17]:
torch.manual_seed(0); random.seed(0); np.random.seed(0)
tcfg = Cfg(pretrained_backbone=False)
model = APGCC(tcfg).to(device)
criterion = APGCCCriterion(tcfg).to(device)
print(f'params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M | head in_dim: {model.decoder.ifi.in_dim}')

B = 4
imgs = torch.randn(B, 3, 128, 128, device=device)
targets = []
for n in [5, 0, 12, 30]:                                   # includes an empty crop
    p = torch.rand(n, 2, device=device) * 127
    targets.append({'point': p, 'labels': torch.ones(n, dtype=torch.long, device=device)})

model.train()
out = model(imgs, targets)
print('pred_logits', tuple(out['pred_logits'].shape), '| pred_points', tuple(out['pred_points'].shape))
assert out['pred_points'].shape[1] == (128 // 8) ** 2 * tcfg.K       # H/8 * W/8 * K proposals

n_gt = sum(len(t['point']) for t in targets)
for kind in ('pos', 'neg'):
    for d in out['aux'][kind]:
        assert d['pred_points'].shape == (n_gt, 2)
        # reference point == auxiliary point  ⇒  pred - offset == a, and |a - gt| within the sampling range
        a = d['pred_points'] - d['offset']
        r = (a - d['gt']).abs()
        if kind == 'pos': assert (r <= tcfg.n_pos + 1e-4).all()
        else:             assert ((r >= tcfg.n_pos - 1e-4) | (a <= 1e-3) | (a >= 128 - 1e-2)).all()
print('APG points:', {k: len(v) for k, v in out['aux'].items()}, '×', n_gt, 'GT')

losses = criterion(out, targets)
print({k: round(v.item(), 4) for k, v in losses.items()})
assert all(torch.isfinite(v) for v in losses.values())
losses['loss'].backward()
g_bb = model.encoder.body1[0].weight.grad.abs().sum().item()
g_pe = model.decoder.ifi.pos1.emb.grad.abs().sum().item()
print(f'grad backbone {g_bb:.3e} | grad φ {g_pe:.3e}')
assert g_bb > 0 and g_pe > 0

params: 17.75M | head in_dim: 196


pred_logits (4, 1024, 2) | pred_points (4, 1024, 2)
APG points: {'pos': 2, 'neg': 2} × 47 GT
{'loss_cls': 0.73, 'loss_loc': 18.9146, 'loss_apg_pos': 1.5558, 'loss_apg_neg': 1.5383, 'loss': 1.3526}


grad backbone 4.524e+02 | grad φ 3.411e-02


In [18]:
# IFI property check: querying the exact centre of a feature cell returns that cell's feature.
feat = torch.randn(1, 3, 4, 4)
pe = SpatialEncoding(2, 8)
centres = grid_query_coords(4, 4, 'cpu')[None]
mix = implicit_sample(feat, centres, pe, mode='paper')[..., pe.out_dim:]
mix = mix.permute(0, 2, 1).reshape(1, 3, 4, 4)
# Exact everywhere except the last row/col: there the +½-cell shift is clamped back into the same cell,
# both neighbours along that axis coincide, all four areas are 0 and weights fall back to 1/4 (same as LIIF/repo).
print('interior exact:', torch.allclose(mix[..., :3, :3], feat[..., :3, :3], atol=1e-4))
assert torch.allclose(mix[..., :3, :3], feat[..., :3, :3], atol=1e-4)

# Off-grid query differs between modes (repo keeps one neighbour only)
q = torch.tensor([[[0.1, -0.3]]])
print('paper vs repo differ:', not torch.allclose(implicit_sample(feat, q, pe, 'paper'),
                                                  implicit_sample(feat, q, pe, 'repo', target_hw=(4, 4))))

interior exact: True
paper vs repo differ: True


In [19]:
# Inference path on a non-multiple-of-16 image + metrics + IR helper
img = torch.randn(1, 3, 250, 333)
pts, scores = predict(model, img, threshold=0.5)
print('predicted count (untrained, meaningless):', len(pts))
gt = torch.rand(20, 2) * torch.tensor([333., 250.])
print('loc counts (tp, fp, fn) @σ=8:', localization_counts(gt + torch.randn_like(gt), gt, 8))

tr = InstabilityTracker()
model.train()
for epoch in range(2):
    out = model(imgs, targets); criterion(out, targets)
    tr.update(range(B), criterion.last_indices)
    print('IR epoch', epoch, tr.end_epoch())

predicted count (untrained, meaningless): 3834
loc counts (tp, fp, fn) @σ=8: (20, 0, 0)


IR epoch 0 nan


IR epoch 1 0.9787234042553191
